# Adding ClinVar benign variants as additional negatives

Goal: expand the negative class (previously only 250 OncoKB negatives, 35 in complete-case).

Label rule:
- Positive (1): OncoKB Oncogenic or Likely Oncogenic
- Negative (0): OncoKB Likely Neutral OR ClinVar Benign / Likely benign

Restricted to missense variants, since the tools only score missense.
Note: VEST4, REVEL, MutPred, VARITY were trained on ClinVar, so their scores are inflated against ClinVar benigns.

In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("exonic_toolscores_oncokb_apicall.txt", sep="\t", low_memory=False)
df = df[df["ANNOTATED"] == True]

tool_cols = ["VEST4_score","REVEL_score","MutPred_score","PrimateAI_score",
             "VARITY_R_score","VARITY_ER_score","ESM1b_score","EVE_score","AlphaMissense_score"]
print("rows:", len(df))

rows: 169360


In [2]:
clinvar = pd.read_csv("hersh_exome_biallelic_annovar_annotated.hg38_multianno.missense_ref_alt_chrom_position_gene_clinvar.tsv", sep="\t", low_memory=False)
print(clinvar.columns.tolist())
print(clinvar["ClinVar_CLNSIG"].value_counts(dropna=False))

['Ref', 'Alt', 'Chrom', 'Position', 'Gene', 'ClinVar_CLNSIG']
ClinVar_CLNSIG
.                                                                       47210
Uncertain_significance                                                  41003
Conflicting_classifications_of_pathogenicity                            10983
Likely_benign                                                            1183
Benign                                                                    764
Benign/Likely_benign                                                      684
Pathogenic/Likely_pathogenic                                              265
Pathogenic                                                                239
Likely_pathogenic                                                         176
not_provided                                                               70
Uncertain_significance/Uncertain_risk_allele                               20
Likely_risk_allele                                               

In [3]:
clv = clinvar[["Chrom","Position","Ref","Alt","ClinVar_CLNSIG"]].rename(
        columns={"Chrom":"Chr","Position":"Start"})

clv = clinvar[["Chrom","Position","Ref","Alt","ClinVar_CLNSIG"]].rename(
        columns={"Chrom":"Chr","Position":"Start"})
df = df.merge(clv, on=["Chr","Start","Ref","Alt"], how="left")
print("variants with a ClinVar call:", df["ClinVar_CLNSIG"].notna().sum())

variants with a ClinVar call: 102603


In [4]:
benign = ["Benign","Likely_benign","Benign/Likely_benign"]

pos = df["ONCOGENIC"].isin(["Oncogenic","Likely Oncogenic"])
neg = (df["ONCOGENIC"] == "Likely Neutral") | (df["ClinVar_CLNSIG"].isin(benign))
missense = df["ExonicFunc.refGeneWithVer"] == "nonsynonymous SNV"

lab = df[missense & (pos | neg)].copy()
lab["label"] = pos[lab.index].astype(int)

for c in tool_cols:
    lab[c] = pd.to_numeric(lab[c], errors="coerce")

print(lab["label"].value_counts())
print("neg from OncoKB :", ((lab.label==0) & (lab["ONCOGENIC"]=="Likely Neutral")).sum())
print("neg from ClinVar:", ((lab.label==0) & (lab["ClinVar_CLNSIG"].isin(benign))).sum())

label
0    2751
1     824
Name: count, dtype: int64
neg from OncoKB : 219
neg from ClinVar: 2585


In [5]:
results = []
for c in tool_cols:
    s = lab[[c, "label"]].dropna()
    scores = s[c].values
    if c == "ESM1b_score":
        scores = -scores
    y = s["label"].values
    results.append({
        "tool": c, "n": len(s),
        "n_pos": int(y.sum()), "n_neg": int((y == 0).sum()),
        "AUROC": round(roc_auc_score(y, scores), 3),
        "AUPRC": round(average_precision_score(y, scores), 3),
    })
res = pd.DataFrame(results).sort_values("AUROC", ascending=False).reset_index(drop=True)
print(res)

                  tool     n  n_pos  n_neg  AUROC  AUPRC
0       VARITY_R_score  3000    714   2286  0.865  0.734
1      VARITY_ER_score  3000    714   2286  0.840  0.688
2          REVEL_score  3428    813   2615  0.826  0.653
3  AlphaMissense_score  3491    814   2677  0.823  0.696
4          VEST4_score  3493    814   2679  0.821  0.645
5          ESM1b_score  3101    736   2365  0.810  0.635
6        MutPred_score  2031    707   1324  0.797  0.714
7            EVE_score  2180    619   1561  0.779  0.643
8      PrimateAI_score  3338    791   2547  0.761  0.516


In [6]:
complete = lab.dropna(subset=tool_cols).copy()
print("complete-case rows:", len(complete))
print(complete["label"].value_counts())

y = complete["label"].values
rows = []
for c in tool_cols:
    scores = complete[c].values
    if c == "ESM1b_score":
        scores = -scores
    rows.append({"tool": c,
                 "AUROC": round(roc_auc_score(y, scores), 3),
                 "AUPRC": round(average_precision_score(y, scores), 3)})
res_cc = pd.DataFrame(rows).sort_values("AUROC", ascending=False).reset_index(drop=True)
print(res_cc)

complete-case rows: 1324
label
0    783
1    541
Name: count, dtype: int64
                  tool  AUROC  AUPRC
0       VARITY_R_score  0.860  0.825
1      VARITY_ER_score  0.831  0.790
2          REVEL_score  0.822  0.785
3  AlphaMissense_score  0.821  0.798
4          VEST4_score  0.819  0.780
5          ESM1b_score  0.816  0.749
6        MutPred_score  0.805  0.767
7            EVE_score  0.793  0.751
8      PrimateAI_score  0.751  0.676


In [7]:
from sklearn.metrics import roc_curve

rows = []
for c in tool_cols:
    s = lab[[c, "label"]].dropna()
    scores = s[c].values.astype(float)
    if c == "ESM1b_score":
        scores = -scores                      # flip so higher = more oncogenic
    y = s["label"].values

    fpr, tpr, thr = roc_curve(y, scores)
    k = (tpr - fpr).argmax()                  # Youden's J: maximize TPR - FPR
    t = thr[k]

    pred = (scores >= t).astype(int)          # flag as oncogenic if score >= cutoff
    tp = ((pred == 1) & (y == 1)).sum()
    fn = ((pred == 0) & (y == 1)).sum()
    fp = ((pred == 1) & (y == 0)).sum()
    tn = ((pred == 0) & (y == 0)).sum()

    rows.append({
        "tool": c,
        "recall":      round(tp / (tp + fn), 3),          # of oncogenic, fraction caught
        "specificity": round(tn / (tn + fp), 3),          # of benign, fraction correctly cleared
        "precision":   round(tp / (tp + fp), 3) if (tp + fp) else float("nan"),
        "youden_J":    round(float((tpr - fpr)[k]), 3),
    })

rec = pd.DataFrame(rows).sort_values("recall", ascending=False).reset_index(drop=True)
print(rec)

                  tool  recall  specificity  precision  youden_J
0       VARITY_R_score   0.744        0.836      0.586     0.580
1  AlphaMissense_score   0.719        0.806      0.530     0.525
2      VARITY_ER_score   0.711        0.816      0.547     0.527
3        MutPred_score   0.702        0.745      0.595     0.446
4          VEST4_score   0.700        0.806      0.523     0.507
5          ESM1b_score   0.686        0.833      0.562     0.520
6            EVE_score   0.685        0.744      0.515     0.429
7      PrimateAI_score   0.675        0.736      0.443     0.411
8          REVEL_score   0.653        0.844      0.566     0.497
